## Objective

This notebook demonstrates a **modern equity research workflow** using **real APIs**:

1. Pull **market prices** and **company fundamentals** via APIs  
2. Clean and align financial time series  
3. Compute valuation, quality, and growth metrics  
4. Perform a simple internal screening assessment  
5. Prepare inputs for valuation (DCF preview)



Company: **Adobe Inc. (ADBE)**  
Fundamentals API: **Alpha Vantage**  
Prices API: **Alpha Vantage**

### 2 - Environment & Import

In [ ]:
import sys
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt

print("Python:", sys.version)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)

### 3- Configuraion

In [34]:
pd.set_option("display.float_format", "{:.4f}".format)
plt.style.use("seaborn-v0_8-whitegrid")

### 4- Parameters & API Setup

In [35]:
ALPHAVANTAGE_API_KEY = "3FOZBB0OVWP3UHYT"
TICKER = "ADBE"

### 5- Alpha Vantage API Helper

   

In [36]:
ALPHA_BASE = "https://www.alphavantage.co/query"

def av_fetch(params):
    params["apikey"] = ALPHAVANTAGE_API_KEY
    r = requests.get(ALPHA_BASE, params=params)
    r.raise_for_status()
    return r.json()

### 6- Load Market Prices (monthly)

In [ ]:
price_raw = av_fetch({
    "function": "TIME_SERIES_MONTHLY_ADJUSTED",
    "symbol": TICKER
})

prices = (
    pd.DataFrame(price_raw["Monthly Adjusted Time Series"])
      .T
      .astype(float)
)

prices.index = pd.to_datetime(prices.index)
prices = prices.sort_index()

prices["Return"] = prices["5. adjusted close"].pct_change()
prices.tail()

### 7- Load Fundamentals - Income Statement

In [ ]:
income_raw = av_fetch({
    "function": "INCOME_STATEMENT",
    "symbol": TICKER
})

income = pd.DataFrame(income_raw["annualReports"])

# Convert date column safely
income["date"] = pd.to_datetime(income["fiscalDateEnding"])
income = income.set_index("date").sort_index()

# Convert all non-text columns to numeric (robust to "None")
for col in income.columns:
    if col not in ["fiscalDateEnding", "reportedCurrency"]:
        income[col] = pd.to_numeric(income[col], errors="coerce")

income.tail()

### 8- Load Balance Sheet

In [ ]:
balance_raw = av_fetch({
    "function": "BALANCE_SHEET",
    "symbol": TICKER
})

balance = pd.DataFrame(balance_raw["annualReports"])


# Convert date column safely
balance["date"] = pd.to_datetime(balance["fiscalDateEnding"])
balance = balance.set_index("date").sort_index()

# Convert all non-text columns to numeric (robust to "None")
for col in balance.columns:
    if col not in ["fiscalDateEnding", "reportedCurrency"]:
        balance[col] = pd.to_numeric(balance[col], errors="coerce")

balance.tail()


### 9- Cash Flow Statement

In [ ]:
cashflow_raw = av_fetch({
    "function": "CASH_FLOW",
    "symbol": TICKER
})

cashflow = pd.DataFrame(cashflow_raw["annualReports"])

# Convert date column safely
cashflow["date"] = pd.to_datetime(cashflow["fiscalDateEnding"])
cashflow = cashflow.set_index("date").sort_index()

# Convert all non-text columns to numeric (robust to "None")
for col in cashflow.columns:
    if col not in ["fiscalDateEnding", "reportedCurrency"]:
        cashflow[col] = pd.to_numeric(cashflow[col], errors="coerce")

cashflow.tail()



###  10 - Core Financial Ratios

In [ ]:
ratios = pd.DataFrame(index=income.index)

# Valuation proxies (illustrative — market cap comes from prices later)
ratios["Revenue"] = income["totalRevenue"]
ratios["Operating Income"] = income["operatingIncome"]
ratios["Net Income"] = income["netIncome"]

# Margins
ratios["Gross Margin"] = income["grossProfit"] / income["totalRevenue"]
ratios["Operating Margin"] = income["operatingIncome"] / income["totalRevenue"]

# Growth
ratios["Revenue Growth"] = income["totalRevenue"].pct_change()
ratios["NI Growth"] = income["netIncome"].pct_change()

ratios.tail()

## Analysis

### 1 - Simple Internal Screening Score

In [ ]:
screen = pd.DataFrame(index=ratios.index)

screen["QualityScore"] = (
    ratios["Operating Margin"].rank()
)

screen["GrowthScore"] = (
    ratios["Revenue Growth"].rank()
)

screen["TotalScore"] = screen["QualityScore"] + screen["GrowthScore"]
screen.sort_values("TotalScore", ascending=False).head()

### 2- Visual Diagnostics
Wealth Index

In [ ]:
wealth = (1 + prices["Return"].dropna()).cumprod()

wealth.plot(
    title="Adobe — Wealth Index (Monthly, Normalised)",
    figsize=(10,5)
)
plt.show()

Revenue & Margin Evolution

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10,8))

income["totalRevenue"].plot(ax=ax[0], title="Revenue")
ratios["Operating Margin"].plot(ax=ax[1], title="Operating Margin")

plt.tight_layout()
plt.show()

### 3- Mini-DCF Prepararatin 

In [ ]:
latest = income.index.max()

revenue_latest = income.loc[latest, "totalRevenue"]
ebit_latest = income.loc[latest, "operatingIncome"]
fcf_latest = cashflow.loc[latest, "operatingCashflow"] - cashflow.loc[latest, "capitalExpenditures"]

print("Latest Revenue:", revenue_latest)
print("Latest EBIT:", ebit_latest)
print("Latest FCF (approx):", fcf_latest)

 ### 4- Research Note (Student Output Template)


## Mini Research Note — Adobe Inc. (ADBE)

**Business Overview:**  
Adobe operates a subscription-based digital media and document platform.

**Quality:**  
High and stable operating margins indicate strong pricing power.

**Growth:**  
Consistent revenue expansion driven by SaaS adoption.

**Risks:**  
- Increased competition (AI‑based design tools)  
- Enterprise IT spending cycles  

**Conclusion:**  
Adobe represents a high‑quality compounder trading at a valuation premium.